In [ ]:
dbutils.widgets.text("account name", "", "account name")
dbutils.widgets.text("container name", "", "container name")
dbutils.widgets.text("account key", "", "account key")
dbutils.widgets.text("file path", "", "file path")

container_name = dbutils.widgets.get("container name")
account_name = dbutils.widgets.get("account name")
destination_file_path = dbutils.widgets.get("file path")
account_key_test = dbutils.widgets.get("account key")

spark.conf.set(f"fs.azure.account.key.{account_name}.blob.core.windows.net", account_key_test)
url = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/{destination_file_path}"

In [ ]:
from pyspark.sql import Row
from delta.tables import DeltaTable


In [ ]:
""" Inital Dataframe to write for delta """
data = [Row(id=1, name="Alice"), Row(id=2, name="Bob")]
df_initial = spark.createDataFrame(data)

df_initial.write.format("delta").mode("overwrite").save(url)


In [ ]:
""" Read delta_table from source """
delta_table = DeltaTable.forPath(spark, url)


In [ ]:
""" Updated Some Value """
incoming_data = [Row(id=1, name="Alicia"), Row(id=3, name="Charlie")]
df_updates = spark.createDataFrame(incoming_data)


In [ ]:
# upsert in delta table
delta_table.alias("target").merge(
    source=df_updates.alias("source"),
    condition="target.id = source.id"
).whenMatchedUpdate(set={"name": "source.name"}) \
 .whenNotMatchedInsert(values={"id": "source.id", "name": "source.name"}) \
 .execute()


In [ ]:
history_df = delta_table.history()